# 04o1: Loading Vector Embeddings into Neo4j

This notebook demonstrates how to load vector embeddings into Neo4j for semantic search.

## Prerequisites

**⚠️ Important:** Before running this notebook, ensure you have:
- Completed [**00-import.ipynb**](./00-import.ipynb) for environment detection and Neo4j connection setup
- Completed [**01-setup-database.ipynb**](./01-setup-database.ipynb#create-vector-index) to create the vector index
- Completed [**02-extracting-embeddings.ipynb**](./02-extracting-embeddings.ipynb) to generate embeddings first
- The vector index must exist before loading embeddings

All environment detection, Neo4j connection, and configuration are handled in `00-import.ipynb`.

## Overview

This notebook stores vector embeddings in Neo4j for semantic search. It takes embeddings generated in [**02-extracting-embeddings.ipynb**](./02-extracting-embeddings.ipynb) and stores them in Neo4j's vector index, enabling semantic search capabilities on your knowledge graph.

**Alternative Approach:** If you prefer storing embeddings in Qdrant instead of Neo4j, see [**04o2-loading-vector-embeddings-qdrant.ipynb**](./04o2-loading-vector-embeddings-qdrant.ipynb) for an alternative vector store.

We'll:
1. Load embeddings generated from [**02-extracting-embeddings.ipynb**](./02-extracting-embeddings.ipynb)
2. Store notes with embeddings in Neo4j vector store

**Note:** The vector index is created in [**01-setup-database.ipynb**](./01-setup-database.ipynb#create-vector-index), and embeddings must be generated in [**02-extracting-embeddings.ipynb**](./02-extracting-embeddings.ipynb) first.


In [ ]:
# Run common imports and setup
%run 00-import.ipynb

# Additional imports specific to this notebook
from knowledge_agents.utils.graph_utils import store_notes_with_embeddings

# Neo4j
from neo4j import GraphDatabase

print("✅ Additional libraries imported")
print("⚠️  Note: This notebook expects embeddings to be generated in 02-extracting-embeddings.ipynb")
print("   If running in the same kernel session, embeddings, file_contents, and file_metadata")
print("   should be available from that notebook.")


In [ ]:
# All setup is done in 00-import.ipynb - no additional configuration needed here


## Load Pre-Generated Embeddings

This notebook expects embeddings to be generated in [**02-extracting-embeddings.ipynb**](./02-extracting-embeddings.ipynb). 

**If running in the same kernel session:** The variables `embeddings`, `file_contents`, and `file_metadata` from notebook 02 should be available here.

**If running in a new kernel session:** You'll need to re-run notebook 02 first, or load saved embeddings.


In [ ]:
# Check if embeddings are available from notebook 02-extracting-embeddings.ipynb
try:
    # Check if variables exist from previous notebook
    if 'embeddings' not in globals() or 'file_contents' not in globals() or 'file_metadata' not in globals():
        raise NameError("Embeddings not found")
    
    print(f"✅ Found embeddings from notebook 02-extracting-embeddings.ipynb")
    print(f"   Number of embeddings: {len(embeddings)}")
    print(f"   Number of files: {len(file_contents)}")
    print(f"   Embedding dimension: {len(embeddings[0]) if embeddings else 0}")
except NameError:
    print("❌ Error: Embeddings not found!")
    print("   Please run 02-extracting-embeddings.ipynb first in the same kernel session,")
    print("   or re-generate embeddings by running that notebook.")
    raise RuntimeError("Embeddings must be generated in 02-extracting-embeddings.ipynb before running this notebook.")


In [ ]:
# Verify embeddings match file contents
if len(embeddings) != len(file_contents):
    raise ValueError(
        f"Mismatch: {len(embeddings)} embeddings but {len(file_contents)} files. "
        "Make sure embeddings and files are from the same run of 02-extracting-embeddings.ipynb"
    )

print(f"✅ Verified: {len(embeddings)} embeddings match {len(file_contents)} files")


## Verify Vector Index

Verify that the vector index exists (created in [**01-setup-database.ipynb**](./01-setup-database.ipynb#create-vector-index)).


In [ ]:
# Verify vector index exists (should have been created in 01-setup-database.ipynb)
with driver.session(database=settings.neo4j_database) as session:
    result = session.run("""
        SHOW INDEXES
        WHERE name = $index_name
    """, index_name=settings.neo4j_vector_index_name)
    
    index_info = result.single()
    if index_info:
        print(f"✅ Vector index '{settings.neo4j_vector_index_name}' exists")
        print(f"   State: {index_info.get('state', 'N/A')}")
        print(f"   Ready for loading embeddings")
    else:
        print(f"❌ Vector index '{settings.neo4j_vector_index_name}' does not exist")
        print(f"   Please run 01-setup-database.ipynb first to create the vector index")
        raise RuntimeError("Vector index must be created before loading embeddings. Run 01-setup-database.ipynb first.")


## Store Notes with Embeddings

Store the notes and their embeddings in Neo4j.


In [ ]:
# Store notes with embeddings in Neo4j using utility function
notes_stored = store_notes_with_embeddings(
    driver=driver,
    embeddings=embeddings,
    file_metadata=file_metadata,
    file_contents=file_contents,
    database=settings.neo4j_database,
    progress_interval=5,  # Print progress every 5 notes
)

print(f"✅ Stored {notes_stored} notes with embeddings in Neo4j")


## Verify Data

Check how many notes are stored with embeddings.


In [ ]:
# Count notes with embeddings
with driver.session(database=settings.neo4j_database) as session:
    result = session.run("""
        MATCH (n:Note)
        WHERE n.embedding IS NOT NULL
        RETURN COUNT(n) as count_with_embeddings
    """)
    
    count = result.single()["count_with_embeddings"]
    print(f"✅ Notes with embeddings: {count}")
    
    # Check total notes
    total_result = session.run("""
        MATCH (n:Note)
        RETURN COUNT(n) as total_notes
    """)
    total = total_result.single()["total_notes"]
    print(f"   Total notes: {total}")


## Next Steps

Now that vector embeddings are loaded, proceed to:
- [**05-querying-graph.ipynb**](./05-querying-graph.ipynb): Query the graph with graph patterns
- [**06-querying-vector-embeddings.ipynb**](./06-querying-vector-embeddings.ipynb): Query using vector similarity search
